# Virtual Browser in Google Colab
This notebook installs a full virtual desktop environment (Xvfb + Fluxbox) along with the Chromium browser, and makes it accessible directly via your web browser using noVNC and LocalTunnel.

In [ ]:
# 1. Install required packages for Desktop GUI, VNC, and Browser
!sudo apt update
!sudo apt install -y xvfb x11vnc fluxbox websockify novnc chromium-browser
!npm install -g localtunnel

In [ ]:
# 2. Start the Virtual Desktop Services
import os
import subprocess
import time

os.environ['DISPLAY'] = ':0'

print("Starting Xvfb...")
xvfb = subprocess.Popen(['Xvfb', ':0', '-screen', '0', '1280x800x24'])
time.sleep(2)

print("Starting Window Manager (Fluxbox)...")
fluxbox = subprocess.Popen(['fluxbox'])
time.sleep(2)

print("Starting VNC Server...")
x11vnc = subprocess.Popen(['x11vnc', '-display', ':0', '-nopw', '-listen', 'localhost', '-xkb', '-ncache', '10', '-ncache_cr', '-forever'])
time.sleep(2)

print("Starting noVNC Web Server on port 8080...")
websockify = subprocess.Popen(['websockify', '--web', '/usr/share/novnc/', '8080', 'localhost:5900'])
time.sleep(2)

print("Starting Chromium Browser...")
chromium = subprocess.Popen(['chromium-browser', '--no-sandbox', '--disable-dev-shm-usage', '--start-maximized', 'https://google.com'])
time.sleep(2)

print("\n✅ Virtual Desktop Services Started Successfully!")

In [ ]:
# 3. Create a public tunnel to access the Desktop
import urllib.request

print("====================================================")
print("🌐 GETTING YOUR TUNNEL URL...")
print("====================================================")

# Start localtunnel
lt = subprocess.Popen(['lt', '--port', '8080'], stdout=subprocess.PIPE)
tunnel_url = lt.stdout.readline().decode('utf-8').strip().replace('your url is: ', '')

# Get Colab IP for LocalTunnel Password
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()

print(f"\n1️⃣ Click this link: {tunnel_url}/vnc.html")
print(f"2️⃣ When asked for the Endpoint IP / Password, enter: {ip}")
print("3️⃣ Click 'Connect' in the noVNC interface to view your browser!")
print("\nLeave this cell running to keep the tunnel alive.")

lt.wait()